In [116]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
#



In [117]:
df = pd.read_csv("qoute_dataset.csv")

In [118]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [119]:
df.shape

(3038, 2)

In [120]:
quotes = df['quote']

In [121]:
import string

In [122]:
quotes = quotes.str.lower()

In [123]:
translator = str.maketrans("","",string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [124]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [125]:
# tokonize
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)


In [126]:
word_index = tokenizer.word_index
print(len(word_index))

8978


In [127]:
sequence = tokenizer.texts_to_sequences(quotes)

In [128]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [129]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [130]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [131]:
max_len = max([len(x) for x in X])
print(max_len)

745


In [132]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen=max_len,padding='pre')


In [133]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [134]:
y= np.array(y)

In [135]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)

In [136]:
y_one_hot.shape

(85271, 10000)

In [137]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN ,LSTM, Dense

In [138]:
emmbedding_dim =50
rnn_units=128



In [139]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=emmbedding_dim,
        input_length=max_len
        )
)

rnn_model.add(
    SimpleRNN(
        units=rnn_units
        )
)
rnn_model.add(
    Dense(
        units=vocab_size,
        activation="softmax"
    )
)

In [140]:
rnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [141]:
rnn_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [142]:
lstm_model=Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size,
    output_dim=emmbedding_dim,
    input_length=max_len
    )
)
lstm_model.add(LSTM(units=vocab_size,activation="softmax"))

In [143]:
lstm_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [144]:
lstm_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [145]:
epochs =10
batch_size=128

In [146]:
# history_rnn = rnn_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split=0.1
# )

## if it run take 2 + hours so ma take a model from cloud

In [147]:
# lstm_model.save("lstm_model.h5")

In [148]:
index_to_words ={}
for word,index in word_index.items():
    index_to_words[index] =word

In [149]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [150]:
from tensorflow.keras.models import load_model

history_rnn = load_model("lstm_model.h5",compile=False)

In [151]:
def predictor(model_ls,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model_ls.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_words[pred_index]

In [155]:
def generate_text(model,tokenizer,seed_text,max_len,n_word):
    for _ in range(n_word):
        next_word = predictor(
                        model,
                        tokenizer,
                        seed_text,
                        max_len
                        )
        if next_word is "":
            break
        seed_text += " " + next_word
    return seed_text

In [159]:
seed = "how are you"
generate_text(history_rnn,tokenizer,seed,max_len,6)

'how are you love to lie down and suck'

In [160]:
import pickle
with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)



In [161]:
with open("max_len.pkl","wb") as f:
    pickle.dump(max_len,f)